# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

In [1]:
import pandas as pd

# Load the dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Clean the column names: lower case and "_" instead of spaces
df.columns = df.columns.str.lower().str.replace(" ", "_", regex=False)

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nColumns with missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Dataset shape: (10910, 26)

Columns: ['unnamed:_0', 'customer', 'state', 'customer_lifetime_value', 'response', 'coverage', 'education', 'effective_to_date', 'employmentstatus', 'gender', 'income', 'location_code', 'marital_status', 'monthly_premium_auto', 'months_since_last_claim', 'months_since_policy_inception', 'number_of_open_complaints', 'number_of_policies', 'policy_type', 'policy', 'renew_offer_type', 'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size', 'vehicle_type']

Columns with missing values:
state                         631
response                      631
months_since_last_claim       633
number_of_open_complaints     633
vehicle_class                 622
vehicle_size                  622
vehicle_type                 5482
dtype: int64


1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [2]:
# Low total claim amount (below 1,000) AND a "Yes" response to the last campaign
df_task1 = df.loc[
    (df["total_claim_amount"] < 1000) &
    (df["response"] == "Yes")
]

print(f"Total customers    : {len(df):,}")
print(f"Filtered customers : {len(df_task1):,} ({len(df_task1) / len(df):.1%})")
df_task1.head()

Total customers    : 10,910
Filtered customers : 1,399 (12.8%)


,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
8,8,FM55990,California,5989.773931,Yes,Premium,College,1/19/11,Employed,M,...,0.0,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN
15,15,CW49887,California,4626.801093,Yes,Basic,Master,1/16/11,Employed,F,...,0.0,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN
19,19,NJ54277,California,3746.751625,Yes,Extended,College,2/26/11,Employed,F,...,1.0,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A
27,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2/28/11,Employed,F,...,0.0,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN


2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [3]:
# Customers who responded "Yes" (from the original DataFrame)
df_yes = df.loc[df["response"] == "Yes"]

# Average premium, customer lifetime value and total claim amount by policy type and gender
segment_summary = (
    df_yes.groupby(["policy_type", "gender"])
    .agg(
        customers=("customer", "count"),
        avg_monthly_premium=("monthly_premium_auto", "mean"),
        avg_clv=("customer_lifetime_value", "mean"),
        avg_total_claim=("total_claim_amount", "mean"),
    )
    .round(2)
)

segment_summary

customers  avg_monthly_premium  avg_clv  \
policy_type    gender                                            
Corporate Auto F             169                94.30  7712.63   
               M             154                92.19  7944.47   
Personal Auto  F             540                99.00  8339.79   
               M             536                91.09  7448.38   
Special Auto   F              35                92.31  7691.58   
               M              32                86.34  8247.09   

                       avg_total_claim  
policy_type    gender                   
Corporate Auto F                433.74  
               M                408.58  
Personal Auto  F                452.97  
               M                457.01  
Special Auto   F                453.28  
               M                429.53

**Conclusions.** Among the 1,466 customers who said "Yes", the claim amounts are quite similar across segments (about 409 to 457 on average), so most of the difference between segments comes from premium and lifetime value.

- **Most valuable large segment: women on Personal Auto.** They pay the highest average premium (99.00), have the highest lifetime value among the large segments (about 8,340) and are the biggest group (540 customers), with an average claim (453) in line with the rest.
- **Lowest-risk segment: men on Corporate Auto.** They have the lowest average claim (about 409) together with a good lifetime value (about 7,944), although their premium is a bit lower than that of women.
- **Weakest segment: men on Personal Auto.** They combine the lowest lifetime value of the large segments (about 7,448), a lower premium (91.09) and the highest average claim (457).
- **Special Auto** looks good for men (lifetime value about 8,247 and claim about 430), but it only has 35 and 32 customers, so this should not be over-interpreted.

Overall, women pay a higher average premium than men in every policy type. Here "profitable" means high premium and lifetime value with low claims; the dataset has no cost data, so this is an indication and not a profit calculation.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [4]:
# Customers per state. The file has repeated customer IDs, so both numbers are shown:
#   customers = number of records (as counted in the lab), unique_customer_ids = distinct IDs
customers_per_state = (
    df.groupby("state")["customer"]
    .agg(customers="count", unique_customer_ids="nunique")
    .sort_values("customers", ascending=False)
)
customers_per_state["share_%"] = (
    customers_per_state["customers"] / customers_per_state["customers"].sum() * 100
).round(1)

# Only the states with MORE THAN 500 customers
states_over_500 = customers_per_state[customers_per_state["customers"] > 500]

print("Customers per state:", len(customers_per_state), "states | states with more than 500 customers:", len(states_over_500))
states_over_500

Customers per state: 5 states | states with more than 500 customers: 5


,customers,unique_customer_ids,share_%
state,,,
California,3552,3150,34.6
Oregon,2909,2601,28.3
Arizona,1937,1703,18.8
Nevada,993,882,9.7
Washington,888,798,8.6


**Conclusions.** All five states have more than 500 customers, so the filter does not remove any of them. California (3,552) and Oregon (2,909) together account for about 63% of the customers, followed by Arizona (1,937), while Nevada (993) and Washington (888) are much smaller. The result is the same when counting unique customer IDs (798 customers in Washington is still above 500). The 631 records without a state are not included.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [5]:
# Maximum, minimum and median customer lifetime value by education level and gender
clv_by_education = (
    df.groupby(["education", "gender"])["customer_lifetime_value"]
    .agg(["max", "min", "median"])
    .round(2)
    .rename(columns={"max": "Max CLV", "min": "Min CLV", "median": "Median CLV"})
)

clv_by_education

Max CLV  Min CLV  Median CLV
education            gender                               
Bachelor             F       73225.96  1904.00     5640.51
                     M       67907.27  1898.01     5548.03
College              F       61850.19  1898.68     5623.61
                     M       61134.68  1918.12     6005.85
Doctor               F       44856.11  2395.57     5332.46
                     M       32677.34  2267.60     5577.67
High School or Below F       55277.45  2144.92     6039.55
                     M       83325.38  1940.98     6286.73
Master               F       51016.07  2417.78     5729.86
                     M       50568.26  2272.31     5579.10

**Conclusions.**

- **A few very large customers drive the maximums.** In every group the maximum is many times the median; the extreme case is men with High School or Below, with a maximum of about 83,325 against a median of about 6,287. The medians are the better measure of a typical customer, and the minimums are almost the same everywhere (about 1,900 to 2,400).
- **More education does not mean more value.** High School or Below has the highest median for both women (about 6,040) and men (about 6,287). The lowest medians are Doctor for women (about 5,332) and Bachelor for men (about 5,548), and Doctor also has the lowest maximums (about 44,856 and 32,677).
- **Gender differences are small and not consistent.** Men have the higher median in High School or Below, College and Doctor; women in Bachelor and Master. The largest gap is in College (about 7%), so gender alone is not a strong basis for targeting.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [6]:
month_names = {1: "January", 2: "February", 3: "March", 4: "April", 5: "May", 6: "June",
               7: "July", 8: "August", 9: "September", 10: "October", 11: "November", 12: "December"}

# Month taken from the policy date (format month/day/year). Pivoting on the month NUMBER keeps the
# columns in calendar order; the names are added afterwards (pivoting on names would be alphabetical).
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], format="%m/%d/%y")
df["month_num"] = df["effective_to_date"].dt.month

policies_by_state_month = (
    df.pivot_table(
        index="state",
        columns="month_num",
        values="policy",
        aggfunc="count",
        fill_value=0,
    )
    .rename(columns=month_names)
    .rename_axis(columns="month")
)

policies_by_state_month

month,January,February
state,,
Arizona,1008,929
California,1918,1634
Nevada,551,442
Oregon,1565,1344
Washington,463,425


**Note.** The data only covers January and February, and the columns now follow calendar order (January, then February). January has more policies than February in every state (5,505 vs 4,774 overall, about 13% fewer in February), and California leads both months.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [7]:
# 1. Group by state and month, count the policies, and sort in descending order
policies_state_month = (
    df.groupby(["state", "month_num"])["policy"]
    .count()
    .reset_index(name="policies_sold")
    .sort_values("policies_sold", ascending=False)
)

# 2. Top 3 states by total policies sold
top3_states = (
    policies_state_month.groupby("state")["policies_sold"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index.tolist()
)
print("Top 3 states:", top3_states)

# 3. New DataFrame with the policies sold by month for those states (states ranked, months in calendar order)
top3_df = (
    policies_state_month[policies_state_month["state"].isin(top3_states)]
    .pivot(index="state", columns="month_num", values="policies_sold")
    .reindex(top3_states)
    .rename(columns=month_names)
    .rename_axis(columns="month")
)
top3_df["total"] = top3_df.sum(axis=1)

top3_df

Top 3 states: ['California', 'Oregon', 'Arizona']


month,January,February,total
state,,,
California,1918,1634,3552
Oregon,1565,1344,2909
Arizona,1008,929,1937


**Note.** California (3,552), Oregon (2,909) and Arizona (1,937) are the top 3 states, and they keep the same order in both months.

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

In [8]:
# Rows without a recorded response cannot be counted as "No", so they are left out
df_resp = df.dropna(subset=["response"])

# melt turns the channel column into a long "channel" variable
# (more channel columns could simply be added to value_vars)
df_melted = df_resp.melt(
    id_vars=["customer", "response"],
    value_vars=["sales_channel"],
    var_name="channel_type",
    value_name="channel",
)
df_melted["responded_yes"] = df_melted["response"] == "Yes"

# Response rate = share of customers who answered "Yes", per channel
response_rate = (
    df_melted.groupby("channel")["responded_yes"]
    .agg(customers="count", yes_responses="sum", response_rate_pct="mean")
    .assign(response_rate_pct=lambda t: (t["response_rate_pct"] * 100).round(2))
    .sort_values("response_rate_pct", ascending=False)
)

print(f"Overall response rate: {df_melted['responded_yes'].mean() * 100:.2f}%")
response_rate

Overall response rate: 14.26%


,customers,yes_responses,response_rate_pct
channel,,,
Agent,3890,742,19.07
Web,1511,177,11.71
Branch,2865,326,11.38
Call Center,2013,221,10.98


**Conclusions.** The overall response rate is 14.26%, and it changes a lot with the channel:

- **Agent is clearly the most effective channel:** 19.07% (742 "Yes" out of 3,890 customers), about 1.6 to 1.7 times the rate of the other channels, and it is also the channel with the most customers.
- **Web, Branch and Call Center are similar** (11.71%, 11.38% and 10.98%): the difference between them is less than one percentage point.

Marketing effort should lean on the Agent channel, while the three lower-performing channels may need a review of their approach. Rows without a recorded response (631) were excluded rather than counted as "No", which would have lowered every rate.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9